In [ ]:
!pip install 'powerlaw'

In [ ]:
import bmll2 as b2
from bmll2 import reference, Security, NormalisedSecurity, SparkHelper, get_market_data, get_market_data_range, VenueMarketError, get_market_tables, save_spark_dataframe, load_spark_dataframe

b2.get_file('modules/metaorder.py')
b2.get_file('modules/LMF.py')

import metaorder as metaorder
import LMF as LMF

import random
import math
import pandas as pd
import polars as pl
import numpy as np
from pandas import StringDtype
from datetime import timedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import LogFormatterSciNotation

from statsmodels.sandbox.stats.runs import runstest_1samp
import powerlaw
import itertools
import pylab
import scipy.stats
import scipy.fft
from scipy.optimize import curve_fit
from sklearn.model_selection import ParameterGrid

from statsmodels.tsa.stattools import acf
from scipy.stats import t

from typing import Literal

In [ ]:
%%time
trader_participation_distribution = 'power'
exchange = 'XJSE'
files = b2.list_files(path = f'data/grid_search_7/{exchange}/{trader_participation_distribution}')

for f in files:
    b2.get_file(f'data/grid_search_7/{exchange}/{trader_participation_distribution}/{f}')

In [ ]:
data = pl.scan_parquet(f'*{trader_participation_distribution}_grid_search.parquet').collect()

if trader_participation_distribution == 'homogenous':
    data.sort(['ticker', 'year', 'N'])
    data.head(10)
if trader_participation_distribution == 'power':
    data.sort(['ticker', 'year', 'N', 'delta'])
    data.head(10)

In [ ]:
# estimate = the estimated value
# expected_value = the expectd_value for the point estimate
# var = the variance of the point estimate
# lambda_penalty = the penalty term that the variance should be multiplied by

def error_calculation(estimate, expected_value, var, lambda_penalty = 1):
    error = np.abs(estimate - expected_value) / np.abs(expected_value) 
    ans   = error + lambda_penalty * (var / expected_value ** 2)
    return ans

In [ ]:
# returns a dataframe that contains the best configuration for each time scale
# data = dataframe containing the configurations details with estimates for the exponents with variances and interval half widths
# config = do you want the configuration that fits the microscopic/macroscopic values the best. Microscopic refers to the impact curve exponent
# the phi exponent and beta
# estimation_method = wether you want the results for when gamma is estimated using the NLLS or the PSD method
# lambda_penatly = the penalty term for the variance
# mix = how the ratio that individual errors for the impact curve exponent, exponent for phi, and beta should be mixed to get the total macroscopic error

def find_configs(data, trader_participation_distribution, config = Literal['metaorder', 'LMF', 'N', 'combined'],
                   estimation_method = Literal['NLLS', 'PSD'], lambda_penalty = 1, mix = [1, 1, 1]):

    impact_error = error_calculation(data[:, 'impact curve exponent'], 0.5, data[:, 'impact curve exponent var'], lambda_penalty)
    phi_error    = error_calculation(data[:, 'phi exponent'], 0.5, data[:, 'phi exponent var'], lambda_penalty)
    gamma        = data[:, f'gamma ({estimation_method.lower()})']
    beta_error   = error_calculation(data[:, 'beta'], 0.5 * (1 - gamma), data[:, 'beta var'], lambda_penalty)
    alpha_error  = np.abs(data[:, 'alpha'] - 1 - gamma) / (gamma + 1)
    N_error      = np.abs(data[:, 'N (st)'] - data[:, f'N ({estimation_method.lower()})']) / data[:, 'N (st)']

    if trader_participation_distribution == 'homogenous':
        error_df = pl.DataFrame({'year': data.select('year'), 'ticker': data.select('ticker'), 'N': data.select('N'),
                                 f'N_{{ST}}': data.select('N (st)'),
                                 f'N_{{est}}': data.select(f'N ({estimation_method.lower()})'),
                                 #f'$\\delta$': data.select('delta'), 
                                 f'$\\alpha$': data.select('alpha'), 
                                 f'$\\gamma$': data.select(f'gamma ({estimation_method.lower()})'),
                                 
                                 f'\\makecell{{$\\frac{{Q}}{{V_D}}$ \\\\ exponent}}': data.select('impact curve exponent'),
                                 f'\\makecell{{$\\phi$ \\\\ exponent}}': data.select('phi exponent'), 
                                 f'$\\beta$': data.select('beta'), 
                                 f'$\\mathrm{{e}}_{{M}}$ ({estimation_method})': (mix[0] * impact_error + mix[1] * phi_error + mix[2] * beta_error) / sum(mix),
                                 f'$\\mathrm{{e}}_{{LMF}}$ ({estimation_method})': alpha_error,
                                 f'$\\mathrm{{e}}_{{N}}$ ({estimation_method})': N_error,
                                 f'$\\mathrm{{e}}_{{N}}$ ({estimation_method})': (mix[0] * impact_error + mix[1] * phi_error + mix[2] * beta_error) / sum(mix) + alpha_error})
    
    if trader_participation_distribution == 'power':
        error_df = pl.DataFrame({'year': data.select('year'), 'ticker': data.select('ticker'), 'N': data.select('N'),
                                 f'N_{{ST}}': data.select('N (st)'),
                                 f'N_{{est}}': data.select(f'N ({estimation_method.lower()})'),
                                 f'$\\delta$': data.select('delta'), 
                                 f'$\\alpha$': data.select('alpha'), 
                                 f'$\\gamma$': data.select(f'gamma ({estimation_method.lower()})'),
                                 
                                 f'\\makecell{{$\\frac{{Q}}{{V_D}}$ \\\\ exponent}}': data.select('impact curve exponent'),
                                 f'\\makecell{{$\\phi$ \\\\ exponent}}': data.select('phi exponent'), 
                                 f'$\\beta$': data.select('beta'), 
                                 f'$\\mathrm{{e}}_{{M}}$ ({estimation_method})': (mix[0] * impact_error + mix[1] * phi_error + mix[2] * beta_error) / sum(mix),
                                 f'$\\mathrm{{e}}_{{LMF}}$ ({estimation_method})': alpha_error,
                                 f'$\\mathrm{{e}}_{{N}}$ ({estimation_method})': N_error,
                                 f'$\\mathrm{{e}}_{{N}}$ ({estimation_method})': (mix[0] * impact_error + mix[1] * phi_error + mix[2] * beta_error) / sum(mix) + alpha_error})
        
    if (config == 'metaorder'):
        error_df = error_df.sort(f'$\\mathrm{{e}}_{{M}}$ ({estimation_method})').group_by(['ticker', 'year']).first()
        error_df = error_df.sort(['ticker', 'year'])
        return error_df

    if (config == 'LMF'):
        error_df = error_df.sort(f'$\\mathrm{{e}}_{{LMF}}$ ({estimation_method})').group_by(['ticker', 'year']).first()
        error_df = error_df.sort(['ticker', 'year'])
        return error_df

    if (config == 'N'):
        error_df = error_df.sort(f'$\\mathrm{{e}}_{{N}}$ ({estimation_method})').group_by(['ticker', 'year']).first()
        error_df = error_df.sort(['ticker', 'year'])
        return error_df

    if (config == 'combined'):
        error_df = error_df.sort(f'$\\mathrm{{e}}_{{N}}$ ({estimation_method})').group_by(['ticker', 'year']).first()
        error_df = error_df.sort(['ticker', 'year'])
        return error_df
        

In [ ]:
trader_participation_distribution = 'power'
config = 'metaorder'
exchange = 'XJSE'
files = b2.list_files(path = f'data/metaorders/{exchange}/calibrated/{trader_participation_distribution}/{config}')

for f in files:
    b2.get_file(f'data/metaorders/{exchange}/calibrated/{trader_participation_distribution}/{config}/{f}')

In [ ]:
all_metaorders = pl.scan_parquet(f'*_XJSE_power_{config}_metaorder_data_calibrated.parquet').collect()

all_metaorders.head(10)

In [ ]:
config = 'N'
estimation_method = 'PSD'

configs = find_configs(data = data, trader_participation_distribution = 'power',
                           config = config, estimation_method = estimation_method,
                           lambda_penalty = 1, mix = [1, 1, 1])#.filter(pl.col('ticker') == 'GRT')

alpha_estimate = configs[f'$\\alpha$'] - 1
gamma_estimate = configs[f'$\\gamma$']

In [ ]:
b2.create_folder('figures_2', ensure_exists = True)
b2.create_folder('figures_2/alpha_gamma', ensure_exists = True)
b2.create_folder(f'figures_2/alpha_gamma/{exchange}', ensure_exists = True)
b2.create_folder(f'figures_2/alpha_gamma/{exchange}/calibrated', ensure_exists = True)
b2.create_folder(f'figures_2/alpha_gamma/{exchange}/calibrated/{config}', ensure_exists = True)

In [ ]:
df = pl.DataFrame({'alpha - 1': alpha_estimate, 'gamma': gamma_estimate})

bins = np.linspace(0, 1, 11)

df = df.with_columns(pl.col('alpha - 1').cut(bins).alias('bin'))

bin_mids = (bins[:-1] + bins[1:]) / 2


plt.figure(figsize = (6, 6))

for i in range(len(bin_mids)):
    
    lower = bins[i]
    upper = bins[i+1]

    gamma_vals = df.filter((pl.col('alpha - 1') >= lower) &(pl.col('alpha - 1') < upper)).select('gamma').drop_nulls().to_series().to_numpy()
    
    if len(gamma_vals) == 0:
        continue

    # Boxplot at midpoint
    plt.boxplot(gamma_vals, positions = [bin_mids[i]], widths = 0.06, patch_artist = True, showfliers = True,
                boxprops = dict(facecolor = 'lightgray', alpha = 0.7)) #showmeans = True, meanline = True

    # Scatter points
    plt.scatter(np.full(len(gamma_vals), bin_mids[i]), gamma_vals, alpha = 0.4, s = 30)

# Theoretical line
plt.plot([0, 1], [0, 1], 'k--', label = r'$\gamma = \alpha - 1$')

plt.xticks(ticks = [0.2, 0.4, 0.6, 0.8, 1.0], labels = ['0.2', '0.4', '0.6', '0.8', '1'])
plt.xlabel(r'$\alpha - 1$')
plt.ylabel(r'$\gamma$')
plt.xlim(0, 1.1)
plt.ylim(0, 1.1)
plt.grid(alpha = 0.3)
plt.legend()
plt.tight_layout()

#plt.savefig(f'{config}_calibrated_alpha_gamma_{estimation_method}.pdf')
#b2.put_file(f'{config}_calibrated_alpha_gamma_{estimation_method}.pdf', f'figures_2/alpha_gamma/{exchange}/calibrated/{config}')
#b2.get_file(f'figures_2/alpha_gamma/{exchange}/calibrated/{config}/{config}_calibrated_alpha_gamma_{estimation_method}.pdf')

plt.show()
